# Connect 4 AlphaZero: Kaggle training session

Thin wrapper around `python -m c4.alphazero.train`. Each session trains for up to
`runtime.max_wall_hours` (default 11h), then saves everything and exits cleanly.
Run it again ("Save Version" -> "Save & Run All") to continue: the run resumes automatically from
the latest W&B checkpoint artifact, or from a previous version's output attached as input.

**One-time setup:** Settings -> Accelerator: GPU (P100 or T4); Internet: on;
Add-ons -> Secrets -> `WANDB_API_KEY`.

In [ ]:
# ---- settings -------------------------------------------------------------------------------
REPO = "https://github.com/Swarsin/Connect-4-Game.git"
REF = "main"                  # pin a tag/commit for reproducible runs
CONFIG = "configs/main.yaml"
OVERRIDES = []                # e.g. ["selfplay.workers=4", "runtime.max_wall_hours=11.5"]
PREV_OUTPUT = None            # e.g. "/kaggle/input/c4-alphazero-train/runs" (previous version output)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc
!rm -rf /kaggle/working/repo && git clone -q {REPO} /kaggle/working/repo && cd /kaggle/working/repo && git checkout -q {REF} && git log -1 --oneline
!pip install -q -e "/kaggle/working/repo[train,eval]"

In [ ]:
import os, shutil, pathlib
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B key loaded")
except Exception as e:
    print("No W&B key (metrics only go to metrics.jsonl):", e)

out = pathlib.Path("/kaggle/working/runs")
out.mkdir(parents=True, exist_ok=True)
if PREV_OUTPUT and pathlib.Path(PREV_OUTPUT).exists():
    shutil.copytree(PREV_OUTPUT, out, dirs_exist_ok=True)
    print("copied previous run outputs from", PREV_OUTPUT)

In [ ]:
sets = " ".join(f"--set {o}" for o in OVERRIDES + ["runtime.out_dir=/kaggle/working/runs"])
!cd /kaggle/working/repo && PYTHONUNBUFFERED=1 python -m c4.alphazero.train --config {CONFIG} --resume auto {sets}

In [ ]:
!ls -la /kaggle/working/runs/*/ /kaggle/working/runs/*/models | head -50
!tail -n 3 /kaggle/working/runs/*/metrics.jsonl